# Session 9: Generalized Linear Models (GLMs)

**Bayesian Analysis of Empirical Data (2026)**  
*Author: Irina Knyazeva*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iknyazeva/bayes-cogsci-book/blob/main/notebooks/colab/09_generalized_linear_models.ipynb)

When outcomes are binary or discrete, regression requires linking the linear predictor $\eta$ to the expected value $\mu$ through an inverse link function. In this laboratory, we model household well-switching in Bangladesh (Gelman et al., *ROS* Ch. 13) and interpret parameters on the natural probability scale.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
import pymc as pm

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
rng = np.random.default_rng(2026)

# Synthetic replica of Bangladesh Arsenic Wells data (N = 500)
N = 500
dist_100m = rng.exponential(0.5, size=N)  # Distance in 100 meters
arsenic = rng.gamma(2.0, 1.0, size=N)     # Arsenic level
logit_p = 0.35 - 0.62 * dist_100m + 0.46 * arsenic
prob_switch = 1.0 / (1.0 + np.exp(-logit_p))
switch_well = rng.binomial(1, prob_switch)

df_wells = pd.DataFrame({"switch": switch_well, "dist_100m": dist_100m, "arsenic": arsenic})
df_wells.head()

,switch,dist_100m,arsenic
0,1,0.074409,0.738007
1,1,0.632865,2.644585
2,1,0.214433,5.985732
3,1,0.343164,0.614365
4,1,0.488881,0.714448


## 1. Fitting Logistic Regression in PyMC

In [2]:
with pm.Model() as wells_model:
    # Priors on log-odds coefficients
    alpha = pm.Normal("alpha", mu=0.0, sigma=1.5)
    beta_dist = pm.Normal("beta_dist", mu=0.0, sigma=1.0)
    beta_ars = pm.Normal("beta_ars", mu=0.0, sigma=1.0)
    
    # Linear predictor and Bernoulli-logit likelihood
    eta = alpha + beta_dist * df_wells["dist_100m"].values + beta_ars * df_wells["arsenic"].values
    y_obs = pm.Bernoulli("switch", logit_p=eta, observed=df_wells["switch"].values)
    
    idata_wells = pm.sample(draws=1000, tune=1000, chains=4, random_seed=2026, return_inferencedata=True)

print(az.summary(idata_wells)[["mean", "sd", "hdi_3%", "hdi_97%", "r_hat"]])

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [alpha, beta_dist, beta_ars]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 1 seconds.


            mean     sd  hdi_3%  hdi_97%  r_hat
alpha      0.609  0.206   0.233    1.003    1.0
beta_dist -0.854  0.179  -1.190   -0.529    1.0
beta_ars   0.412  0.093   0.227    0.580    1.0


## 2. Converting Log-Odds into Natural Probability Contrasts

In [3]:
post = idata_wells.posterior
a = post["alpha"].values.flatten()
b_d = post["beta_dist"].values.flatten()
b_a = post["beta_ars"].values.flatten()

# Probability difference when moving from 0m to 100m away (dist_100m = 0 to 1) at median arsenic = 1.5:
p_0 = 1.0 / (1.0 + np.exp(-(a + b_d * 0.0 + b_a * 1.5)))
p_1 = 1.0 / (1.0 + np.exp(-(a + b_d * 1.0 + b_a * 1.5)))
delta_p = p_1 - p_0

print(f"Probability of switching at 0m:   {p_0.mean():.3f} [95% HDI: {np.percentile(p_0, 2.5):.3f}, {np.percentile(p_0, 97.5):.3f}]")
print(f"Probability of switching at 100m: {p_1.mean():.3f} [95% HDI: {np.percentile(p_1, 2.5):.3f}, {np.percentile(p_1, 97.5):.3f}]")
print(f"\nNet Probability Difference:      {delta_p.mean()*100:.1f} percentage points")

Probability of switching at 0m:   0.772 [95% HDI: 0.721, 0.819]
Probability of switching at 100m: 0.592 [95% HDI: 0.529, 0.650]

Net Probability Difference:      -18.1 percentage points
